# Multi-Horizon Meteorological Time-Series Forecasting with LSTM and Transformer
## JRC MARS Meteorological Database

**Experiment:** Multi-horizon forecasting (1 / 3 / 7 days ahead) for daily temperature prediction

**Dataset:** JRC MARS Meteorological Database, single grid cell (IDCELL: 549573), 16,772 daily observations (1980–2025), chronological 70/15/15 split

**Models:** Climatological Seasonal (baseline) · LSTM · Transformer

**Metrics:** MAE · RMSE

---

In [ ]:
# ── Imports and Setup ────────────────────────────────────────────────────────
!pip install shap -q

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# ── Load Dataset from Google Drive ───────────────────────────────────────────
# ── Load Dataset from Google Drive ───────────────────────────────────────────
from google.colab import drive
import zipfile
import os

drive.mount('/content/drive')

file_path   = '/content/drive/MyDrive/Dataset_Gridded_daily_agr.zip'
extract_dir = 'agri4cast_data'

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(file_path, "r") as z:
    print("Zip contents:")
    print(z.namelist())
    z.extractall(extract_dir)
    print(f"✅ Extracted to folder: {extract_dir}")

csv_files = []
for root, dirs, files_list in os.walk(extract_dir):
    for file in files_list:
        if file.endswith(".csv"):
            csv_files.append(os.path.join(root, file))

print(f"\nCSV files found:")
for f in csv_files:
    print(f"  {f}")

path = csv_files[0]

In [ ]:
# ── Initial Data Inspection (sample of 100,000 rows) ─────────────────────────
import pandas as pd

df = pd.read_csv(path, sep=";", nrows=100000)

print("SHAPE:", df.shape)
print("\nFIRST 3 ROWS:")
print(df.head(3).to_string())
print("\nCOLUMNS:")
print(df.columns.tolist())
print("\nDTYPES:")
print(df.dtypes)
print("\nMISSING VALUES:")
print(df.isnull().sum())

In [ ]:
# ── Grid Cell and Temporal Coverage ──────────────────────────────────────────
print("Unique IDCELLs:", df['IDCELL'].nunique())
print("Unique dates:", df['DAY'].nunique())
print("\nFirst date:", df['DAY'].min())
print("Last date:", df['DAY'].max())
print("\nRows per IDCELL:")
print(df.groupby('IDCELL').size().describe())

In [ ]:
# ── Load Full Dataset for Selected Grid Cell ──────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

feature_cols = ['TEMPERATURE_MAX', 'TEMPERATURE_MIN', 'TEMPERATURE_AVG',
                'WINDSPEED', 'VAPOURPRESSURE', 'PRECIPITATION', 'ET0', 'RADIATION']

chunks = []
for chunk in pd.read_csv(path, sep=';', chunksize=500000):
    best_cell = chunk.groupby('IDCELL').size().idxmax() if len(chunks) == 0 else best_cell
    chunks.append(chunk[chunk['IDCELL'] == best_cell])

df_full = pd.concat(chunks).reset_index(drop=True)
df_full['DAY'] = pd.to_datetime(df_full['DAY'])
df_full = df_full.sort_values('DAY').reset_index(drop=True)

print(f"IDCELL: {best_cell}")
print(f"Total observations: {len(df_full)}")
print(f"Period: {df_full['DAY'].min()} → {df_full['DAY'].max()}")

In [ ]:
# ── Missing Values and Descriptive Statistics ────────────────────────────────
print("=== MISSING VALUES ===")
print(df_full.isnull().sum())

print("\n=== DESCRIPTIVE STATISTICS ===")
print(df_full[feature_cols].describe().round(2))

In [ ]:
# ── Time Series Continuity Check ─────────────────────────────────────────────
date_range = pd.date_range(start=df_full['DAY'].min(),
                            end=df_full['DAY'].max(), freq='D')
missing_dates = date_range.difference(df_full['DAY'])

print(f"Expected days: {len(date_range)}")
print(f"Actual days:   {len(df_full)}")
print(f"Missing days:  {len(missing_dates)}")
if len(missing_dates) > 0:
    print("Missing dates:")
    print(missing_dates)

In [ ]:
# ── Outlier Detection (Z-score > 3) ──────────────────────────────────────────
print("=== OUTLIERS (Z-score > 3) ===")
for col in feature_cols:
    z_scores = np.abs(stats.zscore(df_full[col].dropna()))
    outliers = (z_scores > 3).sum()
    print(f"  {col}: {outliers} outliers ({outliers/len(df_full)*100:.2f}%)")

In [ ]:
# ── Time Series Visualization ────────────────────────────────────────────────
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].plot(df_full['DAY'], df_full[col], linewidth=0.5, color='steelblue')
    axes[i].set_title(col)
    axes[i].set_xlabel('Year')
    axes[i].grid(True, alpha=0.3)

plt.suptitle(f'Time Series Overview — IDCELL {best_cell}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('timeseries_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Correlation Analysis ─────────────────────────────────────────────────────
corr = df_full[feature_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCorrelation with TEMPERATURE_AVG:")
print(corr['TEMPERATURE_AVG'].sort_values(ascending=False).round(3))

In [ ]:
# ── Seasonal Patterns ────────────────────────────────────────────────────────
df_full['month'] = df_full['DAY'].dt.month
monthly = df_full.groupby('month')[feature_cols].mean()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

monthly['TEMPERATURE_AVG'].plot(ax=axes[0,0], marker='o', color='steelblue')
axes[0,0].set_title('Avg Temperature per Month')
axes[0,0].set_xlabel('Month')
axes[0,0].grid(True, alpha=0.3)

monthly['PRECIPITATION'].plot(ax=axes[0,1], marker='o', color='coral')
axes[0,1].set_title('Avg Precipitation per Month')
axes[0,1].set_xlabel('Month')
axes[0,1].grid(True, alpha=0.3)

monthly['ET0'].plot(ax=axes[1,0], marker='o', color='green')
axes[1,0].set_title('Avg ET0 per Month')
axes[1,0].set_xlabel('Month')
axes[1,0].grid(True, alpha=0.3)

monthly['RADIATION'].plot(ax=axes[1,1], marker='o', color='orange')
axes[1,1].set_title('Avg Radiation per Month')
axes[1,1].set_xlabel('Month')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('Seasonal Patterns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Engineering ──────────────────────────────────────────────────────
df_full['day_of_year'] = df_full['DAY'].dt.dayofyear
df_full['month']       = df_full['DAY'].dt.month

# Cyclical encoding for seasonality
df_full['sin_day'] = np.sin(2 * np.pi * df_full['day_of_year'] / 365)
df_full['cos_day'] = np.cos(2 * np.pi * df_full['day_of_year'] / 365)

# Lag features for TEMPERATURE_AVG
df_full['temp_lag1'] = df_full['TEMPERATURE_AVG'].shift(1)
df_full['temp_lag3'] = df_full['TEMPERATURE_AVG'].shift(3)
df_full['temp_lag7'] = df_full['TEMPERATURE_AVG'].shift(7)

# Rolling mean
df_full['temp_roll7']  = df_full['TEMPERATURE_AVG'].rolling(7).mean()
df_full['temp_roll30'] = df_full['TEMPERATURE_AVG'].rolling(30).mean()

# Drop NaN rows from lags/rolling
df_full = df_full.dropna().reset_index(drop=True)

# Update feature_cols
feature_cols = ['TEMPERATURE_MAX', 'TEMPERATURE_MIN', 'TEMPERATURE_AVG',
                'WINDSPEED', 'VAPOURPRESSURE', 'PRECIPITATION', 'ET0', 'RADIATION',
                'sin_day', 'cos_day', 'temp_lag1', 'temp_lag3', 'temp_lag7',
                'temp_roll7', 'temp_roll30']

input_size = len(feature_cols)
print(f"Features: {input_size}")
print(df_full[feature_cols].describe().round(2))

In [ ]:
# ── Data Preparation and Train/Validation/Test Split ─────────────────────────
best_cell = df_full.groupby('IDCELL').size().idxmax()
df_cell = df_full[df_full['IDCELL'] == best_cell].copy()
df_cell = df_cell.sort_values('DAY').reset_index(drop=True)

print(f"IDCELL: {best_cell}")
print(f"Observations: {len(df_cell)}")
print(f"Period: {df_cell['DAY'].min()} → {df_cell['DAY'].max()}")

feature_cols = ['TEMPERATURE_MAX', 'TEMPERATURE_MIN', 'TEMPERATURE_AVG',
                'WINDSPEED', 'VAPOURPRESSURE', 'PRECIPITATION', 'ET0', 'RADIATION',
                'sin_day', 'cos_day', 'temp_lag1', 'temp_lag3', 'temp_lag7',
                'temp_roll7', 'temp_roll30']
target_col  = 'TEMPERATURE_AVG'
target_idx  = feature_cols.index(target_col)
input_size  = len(feature_cols)

data = df_cell[feature_cols].values

n         = len(data)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

# Scaler fitted only on training set
scaler     = MinMaxScaler()
data_train = scaler.fit_transform(data[:train_end])
data_val   = scaler.transform(data[train_end:val_end])
data_test  = scaler.transform(data[val_end:])

print(f"\nTrain: {train_end} | Val: {val_end - train_end} | Test: {n - val_end}")
print(f"data_train shape: {data_train.shape}")
print(f"data_val shape:   {data_val.shape}")
print(f"data_test shape:  {data_test.shape}")

In [ ]:
# ── Dataset Class and Configuration ──────────────────────────────────────────
class TimeSeriesDataset(Dataset):
    def __init__(self, data, seq_len, horizon):
        self.data    = torch.FloatTensor(data)
        self.seq_len = seq_len
        self.horizon = horizon

    def __len__(self):
        return len(self.data) - self.seq_len - self.horizon + 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + self.seq_len + self.horizon - 1, target_idx]
        return x, y

SEQ_LEN    = 30
BATCH_SIZE = 64
HORIZONS   = [1, 3, 7]

print(f"target_idx: {target_idx}")
print(f"input_size: {input_size}")

# Create DataLoaders for horizon=1 as test
ds = TimeSeriesDataset(data_train, SEQ_LEN, 1)
x0, y0 = ds[0]
print(f"Sample x shape: {x0.shape}")
print(f"Sample y shape: {y0.shape}")

In [ ]:
# ── Model Architectures ──────────────────────────────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()


class TransformerModel(nn.Module):
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.input_proj  = nn.Linear(input_size, d_model)
        encoder_layer    = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=128,
                                                       dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.fc          = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.transformer(x)
        return self.fc(x[:, -1, :]).squeeze()

print(f"input_size: {input_size}")

In [ ]:
# ── Training and Evaluation Functions ────────────────────────────────────────
def train_model(model, train_loader, val_loader, epochs=50, patience=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    best_val   = float('inf')
    counter    = 0
    best_state = {k: v.clone() for k, v in model.state_dict().items()}

    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                val_loss += criterion(model(x), y).item()
        val_loss /= len(val_loader)

        if val_loss < best_val:
            best_val   = val_loss
            counter    = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            counter += 1
            if counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    return model


def evaluate_model(model, test_loader, scaler, target_idx):
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for x, y in test_loader:
            preds.extend(model(x.to(device)).cpu().numpy())
            actuals.extend(y.numpy())

    def inv(vals):
        dummy = np.zeros((len(vals), len(feature_cols)))
        dummy[:, target_idx] = vals
        return scaler.inverse_transform(dummy)[:, target_idx]

    preds   = inv(np.array(preds))
    actuals = inv(np.array(actuals))

    mae  = mean_absolute_error(actuals, preds)
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    return mae, rmse, preds, actuals

print("Functions defined!")

In [ ]:
# ── Persistence Baseline ──────────────────────────────────────────────────────
def persistence_baseline(data_test, scaler, target_idx, horizon):
    """Naive baseline: prediction is the value horizon days ago."""
    def inv(vals):
        dummy = np.zeros((len(vals), len(feature_cols)))
        dummy[:, target_idx] = vals
        return scaler.inverse_transform(dummy)[:, target_idx]

    actuals = data_test[horizon:, target_idx]
    preds   = data_test[:-horizon, target_idx]
    actuals = inv(actuals)
    preds   = inv(preds)

    mae  = mean_absolute_error(actuals, preds)
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    return mae, rmse

print("persistence_baseline defined!")

In [ ]:
# ── Experiment Execution ─────────────────────────────────────────────────────
results = {}

for horizon in HORIZONS:
    print(f"\n{'='*50}")
    print(f"HORIZON: {horizon} day(s)")
    results[horizon] = {}

    mae_p, rmse_p = persistence_baseline(data_test, scaler, target_idx, horizon)
    results[horizon]['Persistence'] = {'MAE': mae_p, 'RMSE': rmse_p,
                                        'preds': None, 'actuals': None, 'model': None}
    print(f"  Persistence → MAE: {mae_p:.3f} | RMSE: {rmse_p:.3f}")

    train_ds = TimeSeriesDataset(data_train, SEQ_LEN, horizon)
    val_ds   = TimeSeriesDataset(data_val,   SEQ_LEN, horizon)
    test_ds  = TimeSeriesDataset(data_test,  SEQ_LEN, horizon)

    x0, y0 = train_ds[0]
    print(f"  Input shape: {x0.shape}")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

    for model_name, ModelClass in [('LSTM', LSTMModel), ('Transformer', TransformerModel)]:
        print(f"\n  Training {model_name}...")

In [ ]:
# ── Results Summary Table ────────────────────────────────────────────────────
rows = []
for horizon in HORIZONS:
    for model_name in ['Persistence', 'LSTM', 'Transformer']:
        r = results[horizon][model_name]
        rows.append({'Horizon': f"{horizon}-day", 'Model': model_name,
                     'MAE': round(r['MAE'], 3), 'RMSE': round(r['RMSE'], 3)})

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

In [ ]:
# ── Predicted vs Actual Temperature ─────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

for i, horizon in enumerate(HORIZONS):
    for j, model_name in enumerate(['LSTM', 'Transformer']):
        ax = axes[i][j]
        preds   = results[horizon][model_name]['preds']
        actuals = results[horizon][model_name]['actuals']

        ax.plot(actuals[:100], label='Actual',    color='black',     linewidth=1.5)
        ax.plot(preds[:100],   label='Predicted', color='royalblue', linewidth=1.5, linestyle='--')
        ax.set_title(f'{model_name} — Horizon {horizon}-day')
        ax.set_xlabel('Time Steps')
        ax.set_ylabel('Temperature (°C)')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.suptitle('Predicted vs Actual Temperature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Scatter Plot: Predicted vs Actual ────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(12, 14))

for i, horizon in enumerate(HORIZONS):
    for j, model_name in enumerate(['LSTM', 'Transformer']):
        ax = axes[i][j]
        preds   = results[horizon][model_name]['preds']
        actuals = results[horizon][model_name]['actuals']

        ax.scatter(actuals, preds, alpha=0.3, s=5, color='royalblue')
        min_val = min(actuals.min(), preds.min())
        max_val = max(actuals.max(), preds.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='y=x')
        ax.set_title(f'{model_name} — Horizon {horizon}-day')
        ax.set_xlabel('Actual (°C)')
        ax.set_ylabel('Predicted (°C)')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.suptitle('Scatter Plot: Predicted vs Actual', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('scatter_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Error Comparison: MAE and RMSE per Horizon ───────────────────────────────
horizons_labels = ['1-day', '3-day', '7-day']
pers_mae  = [results[h]['Persistence']['MAE'] for h in HORIZONS]
lstm_mae  = [results[h]['LSTM']['MAE']        for h in HORIZONS]
trans_mae = [results[h]['Transformer']['MAE'] for h in HORIZONS]
pers_rmse  = [results[h]['Persistence']['RMSE'] for h in HORIZONS]
lstm_rmse  = [results[h]['LSTM']['RMSE']        for h in HORIZONS]
trans_rmse = [results[h]['Transformer']['RMSE'] for h in HORIZONS]

x     = np.arange(len(HORIZONS))
width = 0.25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(x - width, pers_mae,  width, label='Persistence', color='gray')
axes[0].bar(x,         lstm_mae,  width, label='LSTM',        color='steelblue')
axes[0].bar(x + width, trans_mae, width, label='Transformer', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(horizons_labels)
axes[0].set_ylabel('MAE (°C)')
axes[0].set_title('MAE per Horizon')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(x - width, pers_rmse,  width, label='Persistence', color='gray')
axes[1].bar(x,         lstm_rmse,  width, label='LSTM',        color='steelblue')
axes[1].bar(x + width, trans_rmse, width, label='Transformer', color='coral')
axes[1].set_xticks(x)
axes[1].set_xticklabels(horizons_labels)
axes[1].set_ylabel('RMSE (°C)')
axes[1].set_title('RMSE per Horizon')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Error Comparison: Persistence vs LSTM vs Transformer', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('error_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SHAP Explainability Analysis: LSTM vs Transformer (Horizon=1) ─────────────
model_shap = results[1]['LSTM']['model'].to('cpu')
model_shap.eval()

test_ds_shap = TimeSeriesDataset(data_test, SEQ_LEN, 1)
sample       = torch.stack([test_ds_shap[i][0] for i in range(100)])
sample_flat  = sample.numpy().reshape(100, -1)
background   = sample_flat[:20]

def model_fn(x):
    x_t = torch.FloatTensor(x).reshape(-1, SEQ_LEN, input_size)
    with torch.no_grad():
        out = model_shap(x_t)
        if out.ndim == 0:
            out = out.reshape(1)
        return out.numpy().reshape(-1, 1)

explainer   = shap.KernelExplainer(model_fn, background)
shap_values = explainer.shap_values(sample_flat[:50], nsamples=100)

sv             = np.array(shap_values).reshape(50, SEQ_LEN, input_size)
mean_shap_lstm = np.abs(sv).mean(axis=(0, 1))

model_shap_t = results[1]['Transformer']['model'].to('cpu')
model_shap_t.eval()

def model_fn_t(x):
    x_t = torch.FloatTensor(x).reshape(-1, SEQ_LEN, input_size)
    with torch.no_grad():
        out = model_shap_t(x_t)
        if out.ndim == 0:
            out = out.reshape(1)
        return out.numpy().reshape(-1, 1)

explainer_t   = shap.KernelExplainer(model_fn_t, background)
shap_values_t = explainer_t.shap_values(sample_flat[:50], nsamples=100)

sv_t            = np.array(shap_values_t).reshape(50, SEQ_LEN, input_size)
mean_shap_trans = np.abs(sv_t).mean(axis=(0, 1))

x     = np.arange(len(feature_cols))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(x - width/2, mean_shap_lstm,  width, label='LSTM',        color='steelblue')
ax.barh(x + width/2, mean_shap_trans, width, label='Transformer', color='coral')
ax.set_yticks(x)
ax.set_yticklabels(feature_cols)
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Feature Importance: LSTM vs Transformer (Horizon=1)')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('shap_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSHAP values per feature (LSTM):")
for f, v in zip(feature_cols, mean_shap_lstm):
    print(f"  {f}: {v:.5f}")

print("\nSHAP values per feature (Transformer):")
for f, v in zip(feature_cols, mean_shap_trans):
    print(f"  {f}: {v:.5f}")

In [ ]:
# ── SHAP Feature Importance — LSTM (Horizon=1) ───────────────────────────────
sv = np.array(shap_values).reshape(50, SEQ_LEN, input_size)
mean_shap_lstm = np.abs(sv).mean(axis=(0, 1))

plt.figure(figsize=(8, 5))
plt.barh(feature_cols, mean_shap_lstm)
plt.xlabel('Mean |SHAP value|')
plt.title('Feature Importance - LSTM (Horizon=1)')
plt.tight_layout()
plt.savefig('shap_lstm.png', dpi=150, bbox_inches='tight')
plt.show()

print("SHAP values per feature:")
for f, v in zip(feature_cols, mean_shap_lstm):
    print(f"  {f}: {v:.5f}")